In [1]:
!pip install chronos-forecasting
!pip install transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
import pandas as pd
import numpy as np
from chronos import Chronos2Pipeline
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.preprocessing import StandardScaler

In [3]:
from huggingface_hub import login
login()

In [4]:
df = pd.read_csv("/content/bitola_final_data.csv")
df.head()

,timestamp,sensorId,city,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,58.333333,28.333333,13.666667,943.0,12.00,8.3,63.645833,...,11.437500,6.98,8.3,6.98,0.000000,1.000000,-0.5,0.866025,0,1
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,13.750000,5.000000,943.0,12.00,9.3,64.750000,...,11.500000,7.50,9.3,7.50,0.258819,0.965926,-0.5,0.866025,0,1
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,60.000000,10.000000,4.500000,942.5,12.25,8.3,65.312500,...,11.625000,6.74,8.3,6.74,0.500000,0.866025,-0.5,0.866025,0,1
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,9.750000,4.500000,942.0,13.00,9.4,64.854167,...,12.166667,7.84,9.4,7.84,0.707107,0.707107,-0.5,0.866025,0,1
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.500000,5.000000,2.000000,942.0,13.00,9.0,65.062500,...,12.312500,8.32,9.0,8.32,0.866025,0.500000,-0.5,0.866025,0,1


In [5]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
count,192984.000000,149772.000000,149781.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,...,192984.000000,192984.000000,192984.000000,192984.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000
mean,52.712150,26.860537,14.470625,939.805608,16.794952,6.075046,52.241040,53.628155,51.232897,943.015291,...,17.258902,6.189260,6.173400,6.157256,-1.384385e-17,-5.550655e-17,-0.002785,-3.554140e-03,0.287278,0.414501
std,15.722700,49.673873,26.783313,12.797334,9.158462,3.743378,15.767958,16.392558,15.040098,6.651617,...,9.219385,3.863181,3.865857,3.848838,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.452493,0.492637
min,9.500000,0.000000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,911.000000,...,-12.000000,0.000000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,0.000000,0.000000
25%,40.759259,5.333333,2.333333,937.000000,9.694444,3.500000,40.466667,41.333333,39.750000,939.000000,...,10.000000,3.500000,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,0.000000,0.000000
50%,53.604167,11.000000,5.750000,942.000000,16.000000,5.300000,53.000000,54.495833,52.250000,943.000000,...,16.370370,5.400000,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,0.000000
75%,64.850000,27.000000,14.500000,946.000000,23.750000,7.740000,64.250000,66.000000,63.000000,947.000000,...,24.083333,7.900000,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,1.000000,1.000000
max,99.000000,1995.000000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,971.750000,...,47.500000,32.800000,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000


In [6]:
len(df)

192984

In [7]:
df['sensorId'].value_counts()

,count
sensorId,
16836a55-7140-43e2-9a63-56fac5cba714,17544
2002,17544
23b735ef-a996-4a7f-9998-2aa7e78827b0,17544
2819ecbb-5de3-4092-aa1d-4ba3a8c40add,17544
40f081a6-4095-43f7-bffb-64e2af8c026e,17544
7b316592-8036-41e2-b8dc-b06b6a9afd54,17544
874ff9c6-786d-45fc-a90e-48c7ffe03417,17544
87f82783-853b-417d-8964-b5cf11e44873,17544
d241a044-0a06-40c2-9d90-c91fd0a95060,17544


In [8]:
TARGET = 'pm10'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 24

In [9]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [10]:
df = df.drop(columns=['pm25','city'],axis=1)

In [11]:
df.columns

Index(['timestamp', 'sensorId', 'humidity', 'pm10', 'pressure', 'temperature',
       'wind_speed', 'neighbor1_humidity', 'neighbor2_humidity',
       'neighbor3_humidity', 'neighbor1_pressure', 'neighbor2_pressure',
       'neighbor3_pressure', 'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'is_weekend', 'is_heating_season'],
      dtype='object')

In [12]:
df.isnull().sum()

,0
timestamp,0
sensorId,0
humidity,0
pm10,43212
pressure,0
temperature,0
wind_speed,0
neighbor1_humidity,0
neighbor2_humidity,0
neighbor3_humidity,0


In [15]:
percentile = np.nanpercentile(df['pm10'], 85)
print("85th percentile of PM10:", percentile)


85th percentile of PM10: 43.5


In [16]:
df['pm10'] = df['pm10'].clip(upper=percentile)

In [17]:
df.describe()

,humidity,pm10,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,neighbor2_pressure,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
count,192984.000000,149772.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,...,192984.000000,192984.000000,192984.000000,192984.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000
mean,52.712150,17.017121,939.805608,16.794952,6.075046,52.241040,53.628155,51.232897,943.015291,942.100966,...,17.258902,6.189260,6.173400,6.157256,-1.384385e-17,-5.550655e-17,-0.002785,-3.554140e-03,0.287278,0.414501
std,15.722700,14.596506,12.797334,9.158462,3.743378,15.767958,16.392558,15.040098,6.651617,6.727275,...,9.219385,3.863181,3.865857,3.848838,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.452493,0.492637
min,9.500000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,911.000000,911.000000,...,-12.000000,0.000000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,0.000000,0.000000
25%,40.759259,5.333333,937.000000,9.694444,3.500000,40.466667,41.333333,39.750000,939.000000,938.000000,...,10.000000,3.500000,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,0.000000,0.000000
50%,53.604167,11.000000,942.000000,16.000000,5.300000,53.000000,54.495833,52.250000,943.000000,942.000000,...,16.370370,5.400000,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,0.000000
75%,64.850000,27.000000,946.000000,23.750000,7.740000,64.250000,66.000000,63.000000,947.000000,946.000000,...,24.083333,7.900000,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,1.000000,1.000000
max,99.000000,43.500000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,971.750000,971.750000,...,47.500000,32.800000,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000


In [18]:
numeric_features = [
    'humidity', 'pressure', 'temperature', 'wind_speed',
    'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
    'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
    'neighbor1_temperature', 'neighbor2_temperature', 'neighbor3_temperature',
    'neighbor1_wind_speed', 'neighbor2_wind_speed', 'neighbor3_wind_speed'
]
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

In [19]:
scalerpm10 = StandardScaler()
df['pm10'] = scalerpm10.fit_transform(df[['pm10']])
df

,timestamp,sensorId,humidity,pm10,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.357521,0.775271,0.249615,-0.523556,0.594372,0.723291,0.611113,0.825325,...,-0.631432,0.204687,0.550100,0.213765,0.000000,1.000000,-0.500000,0.866025,0,1
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.415823,-0.223830,0.249615,-0.523556,0.861511,0.793317,0.678471,0.898740,...,-0.624653,0.339291,0.808775,0.348871,0.258819,0.965926,-0.500000,0.866025,0,1
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.463525,-0.480741,0.210544,-0.496258,0.594372,0.828991,0.712785,0.936140,...,-0.611095,0.142562,0.550100,0.151408,0.500000,0.866025,-0.500000,0.866025,0,1
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.415823,-0.497869,0.171473,-0.414367,0.888225,0.799923,0.684825,0.905666,...,-0.552341,0.427302,0.834643,0.437210,0.707107,0.707107,-0.500000,0.866025,0,1
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.431724,-0.823290,0.171473,-0.414367,0.781370,0.813136,0.697534,0.919518,...,-0.536523,0.551552,0.731172,0.561923,0.866025,0.500000,-0.500000,0.866025,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192979,2025-11-30 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.574830,-0.395104,0.171473,-0.851122,-0.046762,0.714042,2.005296,1.153615,...,-0.974159,-0.230189,-0.070722,-0.222732,-0.965926,0.258819,-0.866025,0.500000,1,1
192980,2025-11-30 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.670233,-0.549251,0.171473,-0.851122,-0.100189,0.729897,2.249310,1.203482,...,-1.028393,-0.204304,-0.122457,-0.196750,-0.866025,0.500000,-0.866025,0.500000,1,1
192981,2025-11-30 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.686134,-0.343722,0.210544,-0.933014,-0.073476,0.703472,2.483156,1.276127,...,-1.038436,-0.204304,-0.096589,-0.196750,-0.707107,0.707107,-0.866025,0.500000,1,1
192982,2025-11-30 22:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.702035,-0.943182,0.249615,-0.987609,-0.126903,0.729897,2.706836,1.306909,...,-1.088652,-0.307846,-0.148324,-0.300677,-0.500000,0.866025,-0.866025,0.500000,1,1


In [20]:
context_df = df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
ground_truth_df = df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)

/tmp/ipykernel_2548/2320177275.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  context_df = df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
/tmp/ipykernel_2548/2320177275.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ground_truth_df = df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)


In [21]:
ground_truth_df

,timestamp,sensorId,humidity,pm10,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,2025-11-30 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.670233,-0.514996,-0.062951,-0.933014,-0.394042,0.634767,0.999756,0.682650,...,-0.787354,-0.566700,-0.407000,-0.404605,0.000000,1.000000,-0.866025,0.5,1,1
1,2025-11-30 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.717935,-0.480741,-0.062951,-0.960311,0.540944,0.603057,0.977388,0.616161,...,-0.787354,-0.540815,0.498364,0.504763,0.258819,0.965926,-0.866025,0.5,1,1
2,2025-11-30 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.558929,-0.686271,-0.023881,-0.960311,0.487517,0.555493,0.931635,0.339123,...,-0.715043,-0.333731,0.446629,0.452799,0.500000,0.866025,-0.866025,0.5,1,1
3,2025-11-30 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.590730,-0.429359,0.015190,-0.960311,0.540944,0.576633,0.979421,0.483183,...,-0.814471,-0.152533,0.498364,0.504763,0.707107,0.707107,-0.866025,0.5,1,1
4,2025-11-30 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.654333,-0.497869,0.015190,-0.960311,0.540944,0.634767,1.011448,0.605080,...,-0.787354,-0.048991,0.498364,0.504763,0.866025,0.500000,-0.866025,0.5,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,2025-11-30 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.574830,-0.395104,0.171473,-0.851122,-0.046762,0.714042,2.005296,1.153615,...,-0.974159,-0.230189,-0.070722,-0.222732,-0.965926,0.258819,-0.866025,0.5,1,1
260,2025-11-30 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.670233,-0.549251,0.171473,-0.851122,-0.100189,0.729897,2.249310,1.203482,...,-1.028393,-0.204304,-0.122457,-0.196750,-0.866025,0.500000,-0.866025,0.5,1,1
261,2025-11-30 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.686134,-0.343722,0.210544,-0.933014,-0.073476,0.703472,2.483156,1.276127,...,-1.038436,-0.204304,-0.096589,-0.196750,-0.707107,0.707107,-0.866025,0.5,1,1
262,2025-11-30 22:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.702035,-0.943182,0.249615,-0.987609,-0.126903,0.729897,2.706836,1.306909,...,-1.088652,-0.307846,-0.148324,-0.300677,-0.500000,0.866025,-0.866025,0.5,1,1


In [22]:
ground_truth_df['timestamp'].max()

Timestamp('2025-11-30 23:00:00+0000', tz='UTC')

In [23]:
ground_truth_df['sensorId'].value_counts()

,count
sensorId,
16836a55-7140-43e2-9a63-56fac5cba714,24
2002,24
23b735ef-a996-4a7f-9998-2aa7e78827b0,24
2819ecbb-5de3-4092-aa1d-4ba3a8c40add,24
40f081a6-4095-43f7-bffb-64e2af8c026e,24
7b316592-8036-41e2-b8dc-b06b6a9afd54,24
874ff9c6-786d-45fc-a90e-48c7ffe03417,24
87f82783-853b-417d-8964-b5cf11e44873,24
d241a044-0a06-40c2-9d90-c91fd0a95060,24


In [24]:
print("Loading Chronos-2 and generating forecasts...")
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="auto",
    dtype=torch.bfloat16,
)

Loading Chronos-2 and generating forecasts...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

In [25]:
future_df = ground_truth_df.drop(columns='pm10',axis=1)

In [26]:
forecast_df = pipeline.predict_df(
    df=context_df,
    prediction_length=PREDICTION_LENGTH,
    target=TARGET,
    id_column=ID_COL,
    future_df = future_df
)

In [27]:
# Merge predictions with actual values to align them
eval_df = pd.merge(
    forecast_df[[ID_COL, TIME_COL, 'predictions']],
    ground_truth_df[[ID_COL, TIME_COL, TARGET]],
    on=[ID_COL, TIME_COL]
)

In [28]:
eval_df.columns

Index(['sensorId', 'timestamp', 'predictions', 'pm10'], dtype='object')

In [29]:
eval_df

,sensorId,timestamp,predictions,pm10
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 00:00:00+00:00,-0.531250,-0.514996
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 01:00:00+00:00,-0.625000,-0.480741
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 02:00:00+00:00,-0.660156,-0.686271
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 03:00:00+00:00,-0.675781,-0.429359
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 04:00:00+00:00,-0.621094,-0.497869
...,...,...,...,...
259,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 19:00:00+00:00,-0.773438,-0.395104
260,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 20:00:00+00:00,-0.839844,-0.549251
261,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 21:00:00+00:00,-0.882812,-0.343722
262,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 22:00:00+00:00,-0.953125,-0.943182


In [30]:
eval_df['predictions'] = scalerpm10.inverse_transform(eval_df[['predictions']])
eval_df['pm10'] = scalerpm10.inverse_transform(eval_df[['pm10']])

In [31]:
eval_df

,sensorId,timestamp,predictions,pm10
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 00:00:00+00:00,9.262753,9.50
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 01:00:00+00:00,7.894336,10.00
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 02:00:00+00:00,7.381179,7.00
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 03:00:00+00:00,7.153110,10.75
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 04:00:00+00:00,7.951353,9.75
...,...,...,...,...
259,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 19:00:00+00:00,5.727674,11.25
260,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 20:00:00+00:00,4.758378,9.00
261,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 21:00:00+00:00,4.131186,12.00
262,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 22:00:00+00:00,3.104872,3.25


In [32]:
eval_df.isnull().sum()

,0
sensorId,0
timestamp,0
predictions,0
pm10,0


In [33]:
y_true = eval_df['pm10']
y_pred = eval_df['predictions']

In [34]:
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"--- Global Model Evaluation ---")
print(f"RMSE: {rmse:.4f}")
print(f"R2 score: {r2:.4f}")

--- Global Model Evaluation ---
RMSE: 9.4170
R2 score: 0.5747


In [35]:
import joblib

feature_scaler = scaler
pm10_scaler = scalerpm10

joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(pm10_scaler, "pm10_scaler.pkl")


['pm10_scaler.pkl']

In [36]:
pipeline.save_pretrained("my_chronos_pipeline_pm10")